In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os

from utils.tools_optimize import TratamentoIniciaisDF# rodar_pipeline_completo
from modelagem_00 import *

c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook onde será testado o modelo SARIMAX
----
- d (ordem de diferenciação)     → 1, confirmado pelo ADF
- p (ordem autoregressiva)       → sugerido pelo PACF: lag 1 é o mais forte,
                                   lag 3-4 também relevantes → começar com p=1 ou p=4
- q (ordem de médias móveis)     → não testei ACF de forma explícita para MA;
                                   começar com q=1 e deixar o AIC decidir
- s (período sazonal)            → lag 27 sugeriu sazonalidade semestral (~26 semanas)
                                   mas o padrão mais robusto de sazonalidade em
                                   séries semanais costuma ser anual (s=52)

In [2]:
path = rf"{os.getcwd()}\data\dados_anp_modelado.parquet"

df = pd.read_parquet(path.replace("\\src",''))

In [3]:
# A partir do df (antes das dummies), ou do df_train original antes do get_dummies
serie_sp_glp = (
    df[(df['Estado - Sigla'] == 'SP') & (df['Produto'] == 'GLP')]
    .set_index('dt_week')
    .sort_index()
)

y_treino = serie_sp_glp.loc[:'2025-04', 'price_sale_median']
y_val    = serie_sp_glp.loc['2025-05':'2025-08', 'price_sale_median']
y_teste  = serie_sp_glp.loc['2025-09':'2026-08', 'price_sale_median']

exog_treino = serie_sp_glp.loc[:'2025-04', ['Valor Venda Dolar', 'Close_BZ=F']]
exog_val    = serie_sp_glp.loc['2025-05':'2025-08', ['Valor Venda Dolar', 'Close_BZ=F']]
exog_teste  = serie_sp_glp.loc['2025-09':'2026-08', ['Valor Venda Dolar', 'Close_BZ=F']]

In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

modelo_sarimax = SARIMAX(
    y_treino,
    exog=exog_treino,
    order=(1, 1, 1),           # (p, d, q)
    seasonal_order=(1, 0, 1, 52),  # (P, D, Q, s) — sazonalidade anual
    enforce_stationarity=False,
    enforce_invertibility=False,
)

resultado_sarimax = modelo_sarimax.fit(disp=False)
print(resultado_sarimax.summary())

                                     SARIMAX Results                                      
Dep. Variable:                  price_sale_median   No. Observations:                  863
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 52)   Log Likelihood               -1182.589
Date:                            Mon, 07 Sep 2026   AIC                           2379.178
Time:                                    20:41:06   BIC                           2412.040
Sample:                                07-20-2008   HQIC                          2391.797
                                     - 05-04-2025                                         
Covariance Type:                              opg                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Valor Venda Dolar    -0.1368      0.591     -0.232      0.817      -1.294       1.021
Close_BZ=F    

In [5]:
pred_val = resultado_sarimax.get_forecast(steps=len(y_val), exog=exog_val)
y_pred_val = pred_val.predicted_mean

from sklearn.metrics import mean_absolute_error

mae_sarimax_val = mean_absolute_error(y_val, y_pred_val)
mae_baseline_val = (y_treino.iloc[-1] - y_val).abs().mean()  # baseline: repete o último valor do treino

print(f"MAE baseline (validação): {mae_baseline_val:.4f}")
print(f"MAE SARIMAX (validação):  {mae_sarimax_val:.4f}")

MAE baseline (validação): 3.0406
MAE SARIMAX (validação):  2.9207


In [6]:
# Baseline "semana a semana": previsão da semana t = valor real da semana t-1
baseline_val = y_val.shift(1)
baseline_val.iloc[0] = y_treino.iloc[-1]  # primeira semana da validação usa o último valor do treino

mae_baseline_val_correto = (baseline_val - y_val).abs().mean()
print(f"MAE baseline (lag_1, validação): {mae_baseline_val_correto:.4f}")

MAE baseline (lag_1, validação): 1.2211


In [7]:
# pip install pmdarima --break-system-packages (se ainda não tiver)
from pmdarima import auto_arima

modelo_auto = auto_arima(
    y_treino,
    exogenous=exog_treino,
    seasonal=True,
    m=52,                      # período sazonal (semanal -> anual)
    d=None,                    # deixa o teste de estacionariedade decidir
    D=None,
    start_p=0, max_p=5,
    start_q=0, max_q=5,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    trace=True,                # mostra o progresso da busca
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True,             # busca mais eficiente que grid completo
)

print(modelo_auto.summary())

c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and w

Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[52] intercept   : AIC=2526.714, Time=0.03 sec
 ARIMA(1,1,0)(1,0,0)[52] intercept   : AIC=2476.553, Time=7.56 sec


c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 ARIMA(0,1,1)(0,0,1)[52] intercept   : AIC=2469.619, Time=7.17 sec
 ARIMA(0,1,0)(0,0,0)[52]             : AIC=2530.507, Time=0.02 sec
 ARIMA(0,1,1)(0,0,0)[52] intercept   : AIC=2467.698, Time=0.07 sec


c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

 ARIMA(3,1,4)(0,0,0)[52] intercept   : AIC=2437.900, Time=1.09 sec


c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 ARIMA(5,1,4)(0,0,0)[52] intercept   : AIC=2438.395, Time=1.23 sec


c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 ARIMA(4,1,3)(0,0,0)[52]             : AIC=2435.431, Time=0.52 sec

Best model:  ARIMA(4,1,3)(0,0,0)[52] intercept
Total fit time: 915.440 seconds
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  863
Model:               SARIMAX(4, 1, 3)   Log Likelihood               -1208.074
Date:                Mon, 07 Sep 2026   AIC                           2434.148
Time:                        21:02:51   BIC                           2476.981
Sample:                    07-20-2008   HQIC                          2450.544
                         - 05-04-2025                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.0080      0.008      0.955      0.340      -0.

In [8]:
pred_val_auto = modelo_auto.predict(n_periods=len(y_val), exogenous=exog_val)

mae_sarimax_auto = mean_absolute_error(y_val, pred_val_auto)

print(f"MAE baseline (lag_1):       {mae_baseline_val_correto:.4f}")
print(f"MAE SARIMAX (auto_arima):   {mae_sarimax_auto:.4f}")

melhora = (1 - mae_sarimax_auto / mae_baseline_val_correto) * 100
print(f"Melhora sobre baseline: {melhora:+.1f}%")

MAE baseline (lag_1):       1.2211
MAE SARIMAX (auto_arima):   2.3865
Melhora sobre baseline: -95.4%


c:\Users\ferna\anaconda3\envs\main\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [10]:
y_treino_diff = y_treino.diff().dropna()

modelo_simples = SARIMAX(
    y_treino_diff,
    order=(1, 0, 1),
    seasonal_order=(0, 0, 0, 0),  # sem sazonalidade, para simplificar
    enforce_stationarity=False,
)
resultado_simples = modelo_simples.fit(disp=False)

pred_diff = resultado_simples.get_forecast(steps=len(y_val))
pred_val_reconstruida = y_treino.iloc[-1] + pred_diff.predicted_mean.cumsum()

mae_simples = mean_absolute_error(y_val, pred_val_reconstruida)
print(f"MAE SARIMAX simples (d manual): {mae_simples:.4f}")

MAE SARIMAX simples (d manual): 2.9678


## Teste com SARIMAX — GLP em São Paulo (série piloto)

### Motivação

Após a conclusão de que modelos baseados em árvore (XGBoost, LightGBM) não
superam o baseline de persistência em nenhuma das 8 combinações testadas
(seção anterior), testou-se SARIMAX como alternativa: diferente de árvores,
modelos ARIMA/SARIMAX projetam tendência de forma explícita e paramétrica,
podendo em tese extrapolar melhor além do range de valores observado no
treino — a limitação central identificada nos modelos anteriores.

### Especificações testadas

| Especificação | Ordem | MAE validação | Baseline (mesmo período) |
|---|---|---|---|
| Manual, com exógenas (dólar, Brent) | SARIMAX(1,1,1)(1,0,1)[52] | 2.92¹ | 3.04¹ |
| `auto_arima` (seleção por AIC) | SARIMAX(4,1,3)(0,0,0)[52] | 2.39 | 1.22 |
| Simplificada, diferenciação manual | SARIMAX(1,0,1), sem sazonalidade | 2.97 | 1.22 |

¹ *Primeira comparação usou baseline calculado incorretamente (persistência
do último valor do treino ao longo de toda a validação, não semana a
semana) — corrigido nas versões seguintes para ser equivalente ao `lag_1`
usado na avaliação dos modelos de árvore.*

### Diagnóstico

Em todas as especificações, os testes de resíduo indicaram problemas
severos de ajuste:

- **Jarque-Bera:** p ≈ 0.00 em todos os casos — resíduos fortemente não-normais
- **Heterocedasticidade:** p ≈ 0.00 — variância do erro não é constante ao longo do tempo
- **Skewness:** entre 1.99 e 2.19 — assimetria pronunciada
- **Kurtosis:** entre 21.35 e 22.99 — caudas extremamente pesadas (muito acima do valor de referência ≈ 3 de uma distribuição normal)

A maior parte dos coeficientes (incluindo as variáveis exógenas dólar e
Brent) não foi estatisticamente significativa nas versões com exógenas,
mesmo essas variáveis tendo correlação de Spearman forte e validada
anteriormente com o preço.

### Conclusão

Em nenhuma das três especificações o SARIMAX aproximou-se do baseline —
pelo contrário, o erro ficou entre 2x e 2.4x maior. Os diagnósticos de
resíduo indicam que a premissa central do modelo (estrutura linear com
resíduos aproximadamente gaussianos) não se sustenta para esta série: o
preço semanal de combustível parece ser mais bem descrito por longos
períodos de persistência quase perfeita, interrompidos por saltos raros e
abruptos, um padrão que modelos lineares clássicos de série temporal não
capturam bem.

Combinado com o resultado da seção anterior, este é o **segundo tipo de
modelo** (após árvores de decisão) que falha em superar o baseline nesta
série — por um motivo estrutural diferente (árvores: dificuldade de
extrapolação além do range de treino; SARIMAX: violação da premissa de
linearidade/normalidade dos resíduos). Isso reforça que a força do baseline
de persistência é uma característica genuína da série, não uma limitação
pontual de uma técnica específica.

### Próximo passo

Testar Prophet como terceira família de modelo, com expectativa calibrada
pelos dois resultados anteriores.